# Module 5: Apache Spark & PySpark

**BigData Analytics Course — AKTN Magister Program**

---

## Learning Objectives
1. Understand the **Spark architecture**: Driver, Executors, Cluster Manager, DAG scheduler.
2. Work with **RDDs** (Resilient Distributed Datasets) — transformations vs actions.
3. Use **Spark DataFrames** and **Spark SQL** for scalable data processing.
4. Apply **MLlib** for distributed machine learning.
5. Handle **streaming** concepts with Structured Streaming.

**Estimated time:** 150 minutes  
**Prerequisites:** Modules 1–4

> ⚠️ **Colab note:** PySpark requires a one-time installation cell. Run it first and wait for completion before continuing.

In [ ]:
# ── Install PySpark (required on Colab / fresh environments) ────────────────
!pip install pyspark findspark --quiet
print("✅ PySpark installed.")

In [ ]:
import findspark
findspark.init()

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    DoubleType, TimestampType
)
from pyspark import SparkContext, SparkConf

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Create a SparkSession (the unified entry point for Spark 2+)
spark = (
    SparkSession.builder
    .appName("BigDataAKTN")
    .master("local[*]")          # use all available CPU cores
    .config("spark.driver.memory", "2g")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

print("Spark version:", spark.version)
print("SparkContext:", spark.sparkContext)
print("✅ SparkSession ready.")

## 1. Spark Architecture

```
┌─────────────────────────────────────────────────────────────────┐
│                        Spark Cluster                            │
│                                                                 │
│  ┌──────────────┐      ┌─────────────────────────────────────┐  │
│  │  Driver      │      │         Worker Nodes                │  │
│  │  Program     │─────▶│  ┌──────────┐   ┌──────────┐       │  │
│  │  (SparkCtx)  │      │  │Executor 1│   │Executor 2│  ...  │  │
│  │              │      │  │Task│Task │   │Task│Task │       │  │
│  └──────────────┘      │  └──────────┘   └──────────┘       │  │
│                        └─────────────────────────────────────┘  │
│              Cluster Manager (YARN / Kubernetes / Standalone)   │
└─────────────────────────────────────────────────────────────────┘
```

Key concepts:
- **Driver:** Your program; creates SparkContext, builds DAG, schedules tasks.
- **Executor:** JVM process on a worker node; executes tasks, stores partitions in memory/disk.
- **DAG (Directed Acyclic Graph):** Spark's lazy execution plan — no computation until an **action** is called.
- **Transformation:** Lazy operation that produces a new RDD/DataFrame (e.g., `filter`, `map`).
- **Action:** Triggers execution and returns a result (e.g., `count`, `collect`, `show`).

## 2. RDDs — Resilient Distributed Datasets

In [ ]:
sc = spark.sparkContext

# ── 2.1 Create an RDD from a Python list ────────────────────────────────────
numbers = sc.parallelize(range(1, 101), numSlices=4)  # distribute across 4 partitions
print(f"Partitions   : {numbers.getNumPartitions()}")
print(f"Total count  : {numbers.count()}")
print(f"First 5      : {numbers.take(5)}")
print(f"Sum          : {numbers.sum()}")
print(f"Mean         : {numbers.mean()}")

In [ ]:
# ── 2.2 Transformations (lazy) ───────────────────────────────────────────────
even_squares = (
    numbers
    .filter(lambda x: x % 2 == 0)        # keep even numbers
    .map(lambda x: x ** 2)               # square them
)
# Nothing has been computed yet — this is a DAG blueprint
print("Type:", type(even_squares))

# ── 2.3 Action — triggers execution ─────────────────────────────────────────
result = even_squares.collect()   # materialises all data in Driver memory
print(f"First 10 even squares: {result[:10]}")
print(f"Sum of even squares  : {even_squares.sum()}")

In [ ]:
# ── 2.4 RDD Word Count (distributed) ────────────────────────────────────────
text_data = [
    "apache spark is a unified analytics engine",
    "spark provides an interface for programming entire clusters",
    "big data processing with apache spark is fast",
    "spark supports java python scala and r",
    "apache spark has a rich ecosystem including spark sql and spark streaming"
]

rdd_text = sc.parallelize(text_data)
word_counts = (
    rdd_text
    .flatMap(lambda line: line.split())                  # tokenise
    .map(lambda word: (word, 1))                         # map: (word, 1)
    .reduceByKey(lambda a, b: a + b)                     # reduce: sum by key
    .sortBy(lambda kv: kv[1], ascending=False)           # sort by count
)

print("Top 10 words:")
for word, count in word_counts.take(10):
    print(f"  {word:<12} {count}")

## 3. Spark DataFrames & Spark SQL

DataFrames are the preferred API for most use cases — they are **higher-level** than RDDs and enable Spark's **Catalyst optimizer** for automatic query optimisation.

In [ ]:
# ── 3.1 Create a Spark DataFrame from a Pandas DataFrame ───────────────────
rng = np.random.default_rng(42)
n = 5_000

pdf = pd.DataFrame({
    'order_id'    : range(1, n + 1),
    'category'    : rng.choice(['Electronics', 'Clothing', 'Books', 'Home', 'Sports'], n),
    'region'      : rng.choice(['North', 'South', 'East', 'West'], n),
    'quantity'    : rng.integers(1, 20, n).astype(int),
    'unit_price'  : rng.uniform(5.0, 500.0, n).round(2),
    'discount_pct': rng.choice([0, 5, 10, 15, 20], n).astype(int),
})
pdf['revenue'] = (pdf['quantity'] * pdf['unit_price'] * (1 - pdf['discount_pct'] / 100)).round(2)

# Convert to Spark DataFrame
sdf = spark.createDataFrame(pdf)

print(f"Spark DataFrame — {sdf.count():,} rows × {len(sdf.columns)} columns")
sdf.printSchema()

In [ ]:
# ── 3.2 DataFrame API operations ────────────────────────────────────────────
# Filter, select, derived column, aggregation
(
    sdf
    .filter(F.col('discount_pct') > 0)                                   # only discounted orders
    .withColumn('net_revenue', F.col('revenue') * 0.89)                  # add GST-equivalent col
    .groupBy('category', 'region')
    .agg(
        F.count('order_id').alias('num_orders'),
        F.sum('revenue').alias('gross_revenue'),
        F.round(F.avg('discount_pct'), 2).alias('avg_discount')
    )
    .orderBy(F.desc('gross_revenue'))
    .show(10, truncate=False)
)

In [ ]:
# ── 3.3 Spark SQL — use SQL directly ────────────────────────────────────────
sdf.createOrReplaceTempView("orders")

result = spark.sql("""
    SELECT
        category,
        COUNT(*)                           AS total_orders,
        ROUND(SUM(revenue), 2)             AS total_revenue,
        ROUND(AVG(revenue), 2)             AS avg_revenue,
        ROUND(STDDEV(revenue), 2)          AS std_revenue,
        ROUND(PERCENTILE(revenue, 0.5), 2) AS median_revenue
    FROM orders
    GROUP BY category
    ORDER BY total_revenue DESC
""")

print("Revenue Statistics by Category (via Spark SQL):")
result.show(truncate=False)

In [ ]:
# ── 3.4 Window functions ─────────────────────────────────────────────────────
from pyspark.sql.window import Window

win = Window.partitionBy('category').orderBy(F.desc('revenue'))

sdf_ranked = (
    sdf
    .withColumn('rank', F.rank().over(win))
    .withColumn('revenue_pct_of_cat',
                F.round(
                    F.col('revenue') / F.sum('revenue').over(Window.partitionBy('category')) * 100,
                    2
                ))
    .filter(F.col('rank') <= 3)
    .select('rank', 'order_id', 'category', 'revenue', 'revenue_pct_of_cat')
    .orderBy('category', 'rank')
)

print("Top 3 orders per category (with window functions):")
sdf_ranked.show(20, truncate=False)

## 4. MLlib — Distributed Machine Learning

In [ ]:
from pyspark.ml import Pipeline as SparkPipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler as SparkScaler, StringIndexer
from pyspark.ml.regression import RandomForestRegressor as SparkRFR
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

# ── 4.1 Prepare features ─────────────────────────────────────────────────────
# Index string columns
cat_indexer    = StringIndexer(inputCol='category', outputCol='category_idx')
region_indexer = StringIndexer(inputCol='region', outputCol='region_idx')

# Assemble feature vector
assembler = VectorAssembler(
    inputCols=['quantity', 'unit_price', 'discount_pct', 'category_idx', 'region_idx'],
    outputCol='features'
)

scaler_mllib = SparkScaler(inputCol='features', outputCol='scaled_features')

# Random Forest Regressor
rf_reg = SparkRFR(
    featuresCol='scaled_features',
    labelCol='revenue',
    numTrees=50,
    maxDepth=5,
    seed=42
)

# Build pipeline
ml_pipeline = SparkPipeline(stages=[
    cat_indexer, region_indexer, assembler, scaler_mllib, rf_reg
])

# Split
train_sdf, test_sdf = sdf.randomSplit([0.8, 0.2], seed=42)
print(f"Train: {train_sdf.count():,}  Test: {test_sdf.count():,}")

In [ ]:
# ── 4.2 Train and evaluate ───────────────────────────────────────────────────
model = ml_pipeline.fit(train_sdf)
predictions = model.transform(test_sdf)

evaluator_rmse = RegressionEvaluator(labelCol='revenue', metricName='rmse')
evaluator_r2   = RegressionEvaluator(labelCol='revenue', metricName='r2')

rmse = evaluator_rmse.evaluate(predictions)
r2   = evaluator_r2.evaluate(predictions)
print(f"MLlib Random Forest — RMSE: {rmse:.2f}  R²: {r2:.4f}")

# Show sample predictions
predictions.select('order_id', 'category', 'revenue', 'prediction').show(10)

## 5. Structured Streaming — Concepts

Spark Structured Streaming treats a live data stream as an **unbounded table** that is continuously appended to. You write the same DataFrame API code as for batch processing.

```python
# Conceptual example — reads from Kafka topic
stream_df = (
    spark.readStream
    .format('kafka')
    .option('kafka.bootstrap.servers', 'localhost:9092')
    .option('subscribe', 'orders')
    .load()
)

# Apply transformations (same API as batch)
aggregated = (
    stream_df
    .groupBy(window(col('timestamp'), '5 minutes'), col('category'))
    .agg(sum('revenue').alias('windowed_revenue'))
)

# Write to console (in production: write to Delta Lake, database, etc.)
query = (
    aggregated.writeStream
    .outputMode('update')
    .format('console')
    .trigger(processingTime='10 seconds')
    .start()
)
query.awaitTermination()
```

### Streaming Output Modes

| Mode | Description |
|------|-------------|
| **complete** | Entire result table written each trigger |
| **append** | Only new rows written (for non-aggregated queries) |
| **update** | Only changed rows written (most efficient for aggregations) |

In [ ]:
# ── Simulate micro-batch streaming with rate source ──────────────────────────
# (rate source generates rows at a fixed rate — ideal for demos)
import time

stream = (
    spark.readStream
    .format('rate')
    .option('rowsPerSecond', 100)
    .load()
    .withColumn('category',
                F.element_at(
                    F.array([F.lit(c) for c in ['Electronics', 'Clothing', 'Books']]),
                    (F.col('value') % 3 + 1).cast('int')
                ))
    .withColumn('revenue', (F.rand() * 500).cast('double'))
)

query = (
    stream
    .groupBy('category')
    .agg(F.count('value').alias('events'), F.round(F.sum('revenue'), 2).alias('total_rev'))
    .writeStream
    .outputMode('complete')
    .format('memory')
    .queryName('streaming_agg')
    .start()
)

# Let it run for 5 seconds then query the in-memory sink
time.sleep(5)
print("Streaming aggregation (5-second snapshot):")
spark.sql("SELECT * FROM streaming_agg ORDER BY total_rev DESC").show()
query.stop()
print("Stream stopped.")

## 6. Performance Tuning Tips

| Tip | Explanation |
|-----|-------------|
| **Partition tuning** | `spark.sql.shuffle.partitions = 2×cores` avoids under/over-parallelism |
| **Caching** | `df.cache()` / `df.persist()` — reuse expensive computations |
| **Broadcast joins** | `F.broadcast(small_df)` — avoids shuffle for joins with small tables |
| **Predicate pushdown** | Filter early; Spark's Catalyst pushes filters to the data source |
| **Columnar formats** | Use Parquet or Delta Lake — enables column pruning and compression |
| **Avoid `collect()`** | Collecting large datasets to driver causes OOM; use `show()` or write instead |

In [ ]:
# ── Cache and explain plan ───────────────────────────────────────────────────
sdf.cache()
print("Cached. Explain plan for a simple aggregation:")
sdf.groupBy('category').agg(F.sum('revenue')).explain()

# Stop Spark session cleanly
spark.stop()
print("✅ SparkSession stopped.")

## 7. Summary

| Concept | Detail |
|---------|--------|
| **RDD** | Low-level distributed dataset; transformations are lazy |
| **DataFrame / SQL** | High-level API; Catalyst optimizer enables automatic performance tuning |
| **MLlib** | Distributed ML; same Pipeline API as scikit-learn |
| **Structured Streaming** | Unified batch + streaming; treats stream as unbounded table |

## 📝 Exercises

1. **RDD:** Implement a distributed matrix multiplication using PySpark RDDs.
2. **DataFrames:** Load the [NYC Taxi Trip dataset](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page) (Parquet) into a Spark DataFrame. Find the top-10 pickup locations by average fare amount.
3. **MLlib:** Implement a classification pipeline using `GBTClassifier` to predict high-revenue orders (revenue > median) from the e-commerce dataset.

---
⬅️ [Module 4 — Machine Learning Fundamentals](Module_04_Machine_Learning_Fundamentals.ipynb)  
➡️ [Module 6 — Big Data Storage & Databases](Module_06_Big_Data_Storage_and_Databases.ipynb)